In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate,  RepeatedKFold, GroupKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import scipy.stats as st
from scipy.stats import pearsonr
import joblib
from xgboost import plot_importance
import seaborn as sns
import statsmodels.api as sm
import optuna
from datetime import datetime
import sklearn
import shap
#Data
# ==============================================================================================================================================================
All_Data= pd.read_csv(r"C:\Users\smuthee\sciebo\Registrations\F_Work_Apps\Data_ Analysis Ex\Soil_Charts\SMC_Data_Shared_31_3_2026.csv")
Data_Model= All_Data.sample(frac=1, random_state=42).reset_index(drop=True)

#Features Engineering
Data_Model['SMC_ProbeN'] = Data_Model['SMC_probe']/Data_Model['Porosity']
Data_Model['Depth_N']= Data_Model['Depth']/Data_Model['Depth'].max()
Data_Model['Bulk_density_N']= Data_Model['Bulk_density']/Data_Model['Bulk_density'].max()

# Continous_Regimes
Data_Model['C_dry']=np.clip(1-(Data_Model['SMC_ProbeN']/0.2),0,1)## Adsorbed
Data_Model['C_wet']=np.clip((Data_Model['SMC_ProbeN'] - 0.8) / (1 - 0.8),0,1)##Satiated
Data_Model['C_normal']=np.clip((1- ( Data_Model['C_wet']+Data_Model['C_dry'])), 0, 1)##Funicular
Data_Model['HRI'] = 0*Data_Model['C_dry'] + 0.5*Data_Model['C_normal'] + 1*Data_Model['C_wet']


# Interaction terms

Data_Model['Bk_MMRI'] = Data_Model['Bulk_density']*Data_Model['MMRI']
Data_Model['Depth_Bk'] = Data_Model['Depth']*Data_Model['Bulk_density']
Data_Model['HRI_P']=Data_Model['SMC_probe']*Data_Model['HRI']

# Features & target
X = Data_Model[['Bulk_density_N','Depth_N','HRI_P','Bk_MMRI','Depth_Bk']]

y=np.log(Data_Model['Vol_Lab']/Data_Model['SMC_probe'])

# CV methods
GKF=GroupKFold(n_splits=3)

# Groups
groups=Data_Model['Name']


###Base model
# =============================================================================================================================================
best_params = {'n_estimators': 168, 
             'max_depth': 2, 
             'learning_rate': 0.09894082551759116, 
             'gamma': 0.5602116770920952, 
             'subsample': 0.9285783352557125, 
             'reg_alpha': 0.528870545491422,
             'reg_lambda': 0.9198993387641355, 
             'colsample_bytree': 0.8359575922718898,
             'min_child_weight': 1
                }


model_base = XGBRegressor(
    objective='reg:pseudohubererror',
    **best_params,
    tree_method='hist',
    eval_metric='rmse',
    #monotone_constraints=monotone_constraint,
    random_state=42,
    n_jobs=1
)


y_testf=[]
y_predict=[]
y_predict_bias_c=[]
y_pred_training=[]
y_trainf=[]
R2_list=[]
R2_list_bias=[]
MAE_list=[]
MAE_list_bias=[]
RMSE_list=[]
RMSE_list_bias=[]
R2_list_training=[]
Unique_groups=np.unique(groups)

for itr in range(10):
    
    for train, test in GKF.split(X,y, groups=groups ):
        X_train, X_test=X.loc[train], X.loc[test]
        y_train, y_test=y[train], y[test]
        
        scaler_x = StandardScaler()
        X_train = scaler_x.fit_transform(X_train)
        X_test = scaler_x.transform(X_test)

        scaler_y = StandardScaler()
        y_train = scaler_y.fit_transform(np.array(y_train).reshape(-1,1))
        y_test = scaler_y.transform(np.array(y_test).reshape(-1,1))
       
        model_base.fit(X_train, y_train) 
        y_pred=model_base.predict(X_test)
        y_pred1_=scaler_y.inverse_transform(y_pred.reshape(-1,1))
    
        y_pred1=np.exp(y_pred1_).ravel() *Data_Model.loc[test,'SMC_probe']
        y_predict.extend(y_pred1)

        y_pred_train1=model_base.predict(X_train)
        y_pred_train2_=scaler_y.inverse_transform(y_pred_train1.reshape(-1,1))
        y_pred_train2=np.exp(y_pred_train2_).ravel() * Data_Model.loc[train,'SMC_probe']
        y_pred_training.extend(y_pred_train2)
        
        y_train1_=scaler_y.inverse_transform(y_train.reshape(-1,1))
        y_train1=np.exp(y_train1_).ravel() * Data_Model.loc[train,'SMC_probe']
        y_trainf.extend(y_train1)

        y_test1_=scaler_y.inverse_transform(y_test.reshape(-1,1))
        y_test1=np.exp(y_test1_).ravel() * Data_Model.loc[test,'SMC_probe']
        y_testf.extend(y_test1)
          
        
        ##Loop Metrics
        R2_list.append(r2_score(y_test1, y_pred1))
        R2_list_training.append(r2_score(y_train1,y_pred_train2))
        MAE_list.append(mean_absolute_error(y_test1, y_pred1))                           
        RMSE_list.append(mean_squared_error(y_test1,y_pred1,squared=False))

## Metrics

RMSE=mean_squared_error(y_testf,y_predict,squared=False)
mae=mean_absolute_error(y_testf, y_predict)
R2=r2_score(y_testf, y_predict)

print(f'RMSE_outside: {RMSE :.4f}')
print(f'RMSE: {np.mean(RMSE_list) :.4f} +/- {np.std(RMSE_list) :.4f}' ) 

print(f'mae_outside: {mae :.4f}')
print(f'MAE: {np.mean(MAE_list) :.4f} +/- {np.std(MAE_list) :.4f}' ) 

print(f'R²_Outside_Loop: {R2 :.4f} ' ) ##Outside loop  
print(f'R²: {np.mean(R2_list) :.4f} +/- {np.std(R2_list) :.4f}' ) 

#Confidence intervals

ci_rmse = st.t.interval(0.95, len(RMSE_list)-1, loc=np.mean(RMSE_list), scale=st.sem(RMSE_list))
print(f'RMSE 95% CI: {ci_rmse[0]:.4f}, {ci_rmse[1]:.4f}')
ci_mae = st.t.interval(0.95, len(MAE_list)-1, loc=np.mean(MAE_list), scale=st.sem(MAE_list))
print(f'MAE 95% CI: {ci_mae[0]:.4f}, {ci_mae[1]:.4f}')
ci_r2 = st.t.interval(0.95, len(R2_list)-1, loc=np.mean( R2_list), scale=st.sem( R2_list))
print(f'R² 95% CI: {ci_r2[0]:.4f}, {ci_r2[1]:.4f}')

fig, ax=plt.subplots(1,3, figsize=(6,6))
ax[0].boxplot(RMSE_list)
ax[0].set_xlabel(f'RMSE:{np.mean(RMSE_list):.4f}, 95% CI [{ci_rmse[0]:.4f}, {ci_rmse[1]:.4f}]', fontsize=7)
ax[0].set_ylabel('RMSE Values')

ax[1].boxplot(MAE_list)
ax[1].set_xlabel(f'MAE:{np.mean(MAE_list):.4f}, 95% CI [{ci_mae[0]:.4f}, {ci_mae[1]:.4f}]', fontsize=7)
ax[1].set_ylabel('MAE Value')

ax[2].boxplot(R2_list)
ax[2].set_ylabel('R² Values')
ax[2].set_xlabel(f'R²:{np.mean( R2_list):.2f} 95% CI [{ci_r2[0]:.2f}, {ci_r2[1]:.2f}]', fontsize=7)

ax[1].set_title('Boxplots of Folds Performance Metrics', loc='center', fontsize=10)
plt.tight_layout(pad=2)
plt.show()

print('=============================================================================================================')

##Bootstrapping- Uncertainities, confidences intervals
def bootstrap_grouped_r2(y_test, y_pred, groups, n_boot=1000, ci=0.95, random_state=42):
    rng = np.random.default_rng(random_state)
    
     ##Convert Arrays
    y_test = np.array(y_test)
    y_pred = np.array(y_pred)
    groups = np.array(groups)
    
    unique_groups = np.unique(groups)
    n_groups = len(unique_groups) 
    
    R2_b=[]
    RMSE_b =[] 
    MAE_b = []
    B_y_pred=[]
    B_y_test=[]
    

    for _ in range(int(n_boot)):
        resampled_groups = rng.choice(unique_groups, size=n_groups, replace=True) ## Resample the groups
        
        indices = []
        for g in resampled_groups:
            index = np.where(groups == g)[0]
            indices.extend(index)
            
        #New /resampled test and preds
        y_t_b = y_test[indices]
        y_p_b = y_pred[indices]
        B_y_pred.append(y_p_b)
        B_y_test.append(y_t_b)
        
        ##Looping metrics
        R2_b.append(r2_score(y_t_b, y_p_b))
        RMSE_b.append(mean_squared_error(y_t_b, y_p_b, squared=False))
        MAE_b.append(mean_absolute_error(y_t_b, y_p_b))

   
    alpha = (1 - ci) / 2 * 100
    return {
        "R2": (np.mean(R2_b), np.percentile(R2_b, alpha), np.percentile(R2_b, 100-alpha), R2_b),
        "RMSE": (np.mean(RMSE_b), np.percentile(RMSE_b, alpha), np.percentile(RMSE_b, 100-alpha), RMSE_b),
        "MAE": (np.mean(MAE_b), np.percentile(MAE_b, alpha), np.percentile(MAE_b, 100-alpha), MAE_b),
        'Booted_Data' :(B_y_test, B_y_pred)
        
    }

B_Rs = bootstrap_grouped_r2(y_testf, y_predict, groups )


print('=============================================================================================================')
print(f"R²_std_Bootstrap_model:{B_Rs['R2'][0]:.4f} ± {np.std(B_Rs['R2'][3]):.4f}")
print(f"CI_model: [{B_Rs['R2'][1]:.4f}, {B_Rs['R2'][2]:.4f}]")

print('================================================================================')
print(f"RMSE_std_Bootstrap_model:{B_Rs['RMSE'][0]:.4f} ± {np.std(B_Rs['RMSE'][3]):.4f}")
print(f"CI_model: [{B_Rs['RMSE'][1]:.4f}, {B_Rs['RMSE'][2]:.4f}]")

print('================================================================================')
print(f"MAE_std_Bootstrap_model:{B_Rs['MAE'][0]:.4F} ± {np.std(B_Rs['MAE'][3]):.4f}")
print(f"CI_model: [{B_Rs['MAE'][1]:.4f}, {B_Rs['MAE'][2]:.4f}]")


#Training vs Test Statistics     

R2_training=r2_score(y_trainf, y_pred_training)
R2_test=r2_score(y_testf, y_predict)

print ("========================================================")
print( '                Training vs Test R2                       ')
print("========================================================= ")

print(f'R²_training {R2_training:.4f}, R²_test {R2_test:.4f}')
print(f'R²_training_folds: {np.mean(R2_list_training) :.4f} +/- {np.std(R2_list_training) :.4f}' ) 


###Cross Val Pred-Unbiased Performances

for i in range (2):
    print('========================================================================================================================================================')
print(f'                                                  Cross Val Predict                                                                                ')

X2=scaler_x.fit_transform(X)
y2=scaler_y.fit_transform(np.array(y).reshape(-1,1))

y_pred_all = cross_val_predict(model_base, X2, y2, groups=groups, cv=GKF)

y_pred_all_c=scaler_y.inverse_transform(y_pred_all.reshape(-1,1))

Data_Model['y_pred1'] = y_pred_all_c.ravel() 
Data_Model['y_pred']=np.exp(Data_Model['y_pred1'])* Data_Model['SMC_probe']

Data_Model['bias'] = Data_Model['Vol_Lab'] - Data_Model['y_pred']
y_L=Data_Model['Vol_Lab']

R2_Cv=r2_score(y_L, Data_Model['y_pred'])
RMSE_Cv=mean_squared_error(y_L,Data_Model['y_pred'], squared=False)
MAE_Cv=mean_absolute_error(y_L, Data_Model['y_pred'])

print('========================================================================================================================================================')
print(f'R²_Cv: {R2_Cv:.4f}')
print(f'RMSE_Cv: {RMSE_Cv:.4f}')
print(f'MAE_Cv: {MAE_Cv:.4f}')
print('========================================================================================================================================================')

plt.scatter( Data_Model['y_pred'],Data_Model['bias'], s=2)
plt.axhline(0, color='red')
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.show()      
Data_Model['bias'].mean()

print('=====================================================================Final Model==========================================================================')

Final_Model=XGBRegressor(
    objective='reg:pseudohubererror',
    **best_params,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='rmse',
    random_state=42,
    n_jobs=1
)


Final_Model.fit(X2,y2)
Final_Base_Pred=Final_Model.predict(X2)
Final_Base_Pred1=np.exp(scaler_y.inverse_transform(np.array(Final_Base_Pred).reshape(-1,1)))
Final_Results= Final_Base_Pred1.ravel()*Data_Model['SMC_probe']

F_res=Data_Model['Vol_Lab']- Final_Results
Data_Model['F_R']=Final_Results

R2_F=r2_score(Data_Model['Vol_Lab'], Data_Model['F_R'])
RMSE_F=mean_squared_error(Data_Model['Vol_Lab'], Data_Model['F_R'], squared=False)
MAE_F=mean_absolute_error(Data_Model['Vol_Lab'], Data_Model['F_R'])

print('========================================================================================================================================================')
print(f'R²_F: {R2_F:.4f}')
print(f'RMSE_F: {RMSE_F:.4f}')
print(f'MAE_F: {MAE_F:.4f}')
print('========================================================================================================================================================')


###Residuals Analysis
MBE_F= np.mean(Data_Model['Vol_Lab']- Data_Model['F_R'])
ubrmse=np.sqrt(RMSE_F**2- MBE_F**2)
print(f'MBE:{MBE_F:.4f}')
print(f'ubrmse: {ubrmse:.4f}')

FR=Data_Model['F_R'].values

## Plots
min_p=min(float(np.min(FR)), float(np.min(Data_Model['Vol_Lab'])))
max_p=max(float(np.max(FR)),float(np.max(Data_Model['Vol_Lab'])))

plt.figure(figsize=(6,6))
plt.scatter(Data_Model['Vol_Lab'], FR, color='k', marker='o', s=10)
plt.plot([min_p, max_p], [min_p, max_p], 'b-', linewidth=0.25)
plt.xlabel('SMC_LAB ($m^3$.$m^-3$)')
plt.ylabel('Calibrated_SMC($m^3$.$m^-3$)')
plt.gca().set_aspect('equal')
plt.show()

##Model Saving
Model_Desc={'Base model':Final_Model, 
            'Scaler_x': scaler_x,
            'Scaler_y': scaler_y,            
            'model_type': 'XGBoost- XGBRegressor',
            'Parameters': best_params,
            'pred': X.columns,
            'Predictors':'SMC_Regimes x Probe_SMC, Bulk density-normalised, Depth-normalised, Bulk density x Micro macro aggregates ratio, Bulk density x depth' ,
            'Target Variable': 'log(Lab_SMC -Probe_SMC)',
            'n_samples': 136,
            'Depths': {10,20,30,40},
            'Final_Metrics': {
                'R²':f'{R2_F :.4}',
                'RMSE': f'{RMSE_F :.4f}',
                'MAE' : f'{MAE_F :.4f}',
                'NSE': f'{R2_F:.4f}',
                'MBE': f'{MBE_F :.4f}',
                'UBRMSE': f'{ubrmse :.4f}'
            },
            'Date':  datetime.now().isoformat(),
            'Sklearn_Version': sklearn.__version__,
            'author': 'MSW'
           }
            
joblib.dump(Model_Desc, 'C:/Users/smuthee/sciebo/Registrations/F_Work_Apps/Data_ Analysis Ex/Soil_Charts/F_XGBoost_Model_Shared.pkl')

print(Model_Desc)

In [ ]:
###Evalaution by Depth
R2_depth=[]
RMSE_depth=[]
Bias_depth=[]
MAE_depth=[]
ubrmse_depth=[]

unique_depths = np.unique(Data_Model['Depth'])

for d in unique_depths:
    mask = Data_Model['Depth'] ==d 
    y_true_d = Data_Model.loc[mask,'Vol_Lab']
    y_pred_d = Data_Model.loc[mask,'F_R']
    
    rmse_d = mean_squared_error(y_true_d, y_pred_d, squared=False)
    RMSE_depth.append(rmse_d)
    
    mae_d = mean_absolute_error(y_true_d, y_pred_d)
    MAE_depth.append(mae_d)
    
    bias_d = np.mean(y_pred_d - y_true_d)
    Bias_depth.append(bias_d)
    
    ubrmse_d=np.sqrt(rmse_d**2- bias_d**2)
    ubrmse_depth.append(ubrmse_d)
    
    if len(y_true_d) > 1:
        R2_d = r2_score(y_true_d, y_pred_d)
        R2_depth.append(R2_d)
    else:
        R2_d = np.nan
    
    print(f' {d}, R2: {np.mean(R2_depth):.4f}')

    print(f' {d}, RMSE: {np.mean(RMSE_depth):.4f}')
    
    print(f' {d}, MAE: {np.mean(MAE_depth):.4f}')
    
    print(f' {d}, Bias: {np.mean(Bias_depth):.4f}')
    
    print(f' {d}, ubrmse: {np.mean(ubrmse_depth):.4f}')
    print(f'====================================')
        

fig, ax=plt.subplots(1,5, figsize=(6,6))
ax[0].boxplot(R2_depth)
ax[0].set_xlabel('R2')

ax[1].boxplot(RMSE_depth)
ax[1].set_xlabel('Rmse')

ax[2].boxplot(MAE_depth)
ax[2].set_xlabel('Mae')

ax[3].boxplot(Bias_depth)
ax[3].set_xlabel('Bias')

ax[4].boxplot(ubrmse_depth)
ax[4].set_xlabel('ubrmse')

ax[2].set_title("Boxplots of Performance Metrics across the Depths", fontsize=10, pad=10)
plt.tight_layout(pad=0.5)
plt.show()

In [ ]:
## Residuals Analysis

X_av= Data_Model['F_R']
y_av=Data_Model['Vol_Lab']- Data_Model['F_R']

min_r=min(float(np.min(X_av)), float(np.min(y_av)))
max_r=max(float(np.max(X_av)),float(np.max(y_av)))

plt.figure(figsize=(10,6))
plt.scatter(X_av,y_av, color='r', marker='o', s=10)
plt.axhline(0, color='b', linewidth=1)
plt.xlim()
plt.xlabel('SMC_LAB ($m^3$.$m^-3$)')
plt.ylabel('Residuals ($m^3$.$m^-3$)')
plt.show()

##Normality check
fig, ax=plt.subplots(1,2, figsize=(12,4))
sns.histplot(y_av,ax=ax[0], kde=True)
ax[0].set_xlabel("Residuals ($m^3$.$m^-3$)")
ax[0].set_title("Residual Distribution", fontsize=10)

##Homoscedasticity
sns.residplot(x=X_av, y=Data_Model['Vol_Lab'], ax=ax[1], lowess=True, line_kws={'color': 'red','label': 'Homoscedasticity test' } )
ax[1].set_xlabel('Predicted_SMC ($m^3$.$m^-3$)')
ax[1].set_ylabel('Residuals ($m^3$.$m^-3$)')
ax[1].set_title('Residuals Scatter with Homoscedasticity Test', fontsize=12)
plt.legend(loc='lower right')
plt.tight_layout(pad=3)
plt.show()


In [ ]:
##Shap Analysis
explainer = shap.TreeExplainer(Final_Model)
shap_values = explainer.shap_values(X)
plt.figure(figsize=(6,6))
shap.summary_plot(shap_values, X, plot_type="bar", show=True)

##Beeswarm
plt.figure(figsize=(6,6))
shap.summary_plot(shap_values, X, show=False)
plt.savefig('C:/Users/smuthee/sciebo/Registrations/F_Work_Apps/Data_ Analysis Ex/Soil_Charts/Beeswarm Plots.jpeg',dpi=600)
plt.show()

# Dependence plot for key interaction
fig, ax=plt.subplots(3,2, figsize=(10,8))
axes=ax.flatten()
D= ["HRI_P", "Bulk_density_N", "Depth_Bk", "Bk_MMRI", "Depth_N"]

for ax1, f in zip(axes, D):
    shap.dependence_plot(f,shap_values, X,
                     interaction_index="auto", ax=ax1, show=False )
    ax1.set_xlabel(ax1.get_xlabel(),fontsize=10),
    ax1.set_ylabel(ax1.get_ylabel(), fontsize=10),
    ax1.set_title(f"SHAP Interactions: {f}", fontsize=10)


for j in range(len(D), len(axes)):
    axes[j].set_axis_off()     
plt.subplots_adjust(wspace=1, hspace=1)
plt.show()

In [ ]:
# Soil Sensitivit
Soil_names=Data_Model['Soil Type']
X_plot=X.copy()
X_plot['Depth_N']=Soil_names.astype('category')

fig, ax=plt.subplots(2,2, figsize=(10,6))
axes=ax.flatten()
                     
ft= ["HRI_P", "Bulk_density_N", "Depth_Bk", "Bk_MMRI"] 
                     
for  ax1, f in zip(axes, ft):
    shap.dependence_plot(
    f, 
    shap_values, 
    X_plot,
    ax=ax1,
    interaction_index='Depth_N', 
    cmap=plt.get_cmap('rainbow'),     
    show=False
    )
    ax1.set_xlabel(ax1.get_xlabel(),fontsize=6)
    ax1.set_ylabel(ax1.get_ylabel(),fontsize=6)
    ax1.set_title(f'{f}:Soil_type Impact',fontsize=6)
    
for i in range (len(ft), len(axes)):
            axes[i].set_axis_off()

plt.subplots_adjust(wspace=1, hspace=1)
plt.show()


